# Langfuse Traces

#### Authenticate with AWS

In [ ]:
from dotenv import load_dotenv
loaded = load_dotenv()
print(loaded)

#### Import metrics for evaluation

In [ ]:
import metrics
from metrics import *


for name in dir(metrics):
    val = getattr(metrics, name)
    if isinstance(val, list):
        print(f"{name}:")
        for m in val:
            print(f"  - {m}")
        print()

#### Choose the set of metrics for evaluation

In [ ]:
metrics = all_metrics
print("You've chosen the following metrics for evaluating the Langfuse traces:")
for m in metrics:
    print(f"  - {m}")

#### Select AWS region and Foundation Model

In [ ]:
region = "us-east-1"
print(f"AWS Region: {region}")

#### Install UAEF
`pip install uaef`

In [ ]:
from uaef.data import parse_ground_truth_row
# Import UAEF
from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime
print("✓ UAEF imported successfully!")

## Evaluate a Langfuse Trace

Langfuse is an observability platform for LLM applications. It captures traces containing:
- **Generations**: LLM calls with input/output, token usage, and cost
- **Spans**: Units of work with duration
- **Events**: Point-in-time occurrences

The `LangfuseAdapter` transforms these hierarchical observations into UAEF's canonical `AgentTrace` format.

#### Step 1: Prepare Langfuse trace data

In production, you would fetch this from the Langfuse API. Here we simulate a trace from a weather agent.

In [ ]:
# Step 1: Simulate a Langfuse trace (in production, fetch via Langfuse SDK/API)
langfuse_trace = {
    "trace_id": "lf-trace-001",
    "session_id": "session-langfuse-001",
    "name": "weather-agent-run",
    "timestamp": "2025-01-15T10:00:00Z",
    "metadata": {"environment": "production", "agent_version": "1.0"},
    "observations": [
        {
            "id": "obs-1",
            "trace_id": "lf-trace-001",
            "type": "generation",
            "name": "llm-call-1",
            "input": {"role": "user", "content": "What's the weather in Seattle?"},
            "output": {"content": "Let me check the weather in Seattle for you."},
            "model": "anthropic.claude-3-sonnet",
            "usage": {"input": 25, "output": 15, "total": 40},
            "cost": 0.0012,
            "start_time": "2025-01-15T10:00:01Z",
            "end_time": "2025-01-15T10:00:02Z",
        },
        {
            "id": "obs-2",
            "trace_id": "lf-trace-001",
            "type": "span",
            "name": "tool_get_weather",
            "input": {"location": "Seattle"},
            "output": {"temp": 72, "condition": "sunny", "location": "Seattle"},
            "start_time": "2025-01-15T10:00:02Z",
            "end_time": "2025-01-15T10:00:03Z",
            "parent_observation_id": "obs-1",
        },
        {
            "id": "obs-3",
            "trace_id": "lf-trace-001",
            "type": "generation",
            "name": "llm-call-2",
            "input": {"role": "user", "content": "Tool result: temp=72, condition=sunny"},
            "output": {"content": "The current weather in Seattle is 72°F and sunny."},
            "model": "anthropic.claude-3-sonnet",
            "usage": {"input": 40, "output": 20, "total": 60},
            "cost": 0.0018,
            "start_time": "2025-01-15T10:00:03Z",
            "end_time": "2025-01-15T10:00:04Z",
        },
    ],
}

print(f"Langfuse trace: {langfuse_trace['trace_id']}")
print(f"  Observations: {len(langfuse_trace['observations'])}")
for obs in langfuse_trace["observations"]:
    print(f"    [{obs['type']}] {obs['name']}")

#### Step 2: Transform with the LangfuseAdapter

In [ ]:
# Step 2: Transform Langfuse trace to canonical AgentTrace
from uaef.adapters import LangfuseAdapter

adapter = LangfuseAdapter()
agent_trace = adapter.transform_to_canonical(langfuse_trace)

print(f"✓ Transformed Langfuse trace to AgentTrace")
print(f"  Trace ID: {agent_trace.trace_id}")
print(f"  Messages: {len(agent_trace.messages)}")
print(f"  Tool Calls: {len(agent_trace.tool_calls)}")
print(f"  Framework: {agent_trace.framework}")
print(f"  Input Tokens: {agent_trace.input_tokens}")
print(f"  Output Tokens: {agent_trace.output_tokens}")
if agent_trace.latency:
    print(f"  Latency: {agent_trace.latency:.3f}s")
if agent_trace.cost:
    print(f"  Cost: ${agent_trace.cost:.4f}")

# Print extracted messages
for msg in agent_trace.messages:
    print(f"\n  [{msg.role.value}]: {msg.content[:200]}")

#### Step 3: Evaluate

In [ ]:
# Step 3: Define ground truth and evaluate
from datetime import timezone

ground_truth = GroundTruth(
    expected_output="The weather in Seattle is 72°F and sunny",
    expected_tool_calls=[
        ToolCall(
            name="get_weather",
            arguments={"location": "Seattle"},
            timestamp=datetime.now(timezone.utc)
        )
    ],
    context_documents=["Seattle is a city in Washington state"]
)

result = evaluate(
    trace=agent_trace,
    ground_truth=ground_truth,
    metrics=metrics
)

print(f"\n{'='*50}")
print("LANGFUSE TRACE - EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Overall Score: {result.overall_score:.2f}")
print(f"Passed: {'✓ Yes' if result.passed else '✗ No'}")

print(f"\n{'='*50}")
print("METRIC SCORES BY DIMENSION")

for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} (aggregate score: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        print(f"  {metric.metric_name}: {metric.score:.2f}")

## Batch Evaluation
1. Modify `data/ground-truth.xlsx` with your Ground Truth questions and answers.
2. Run batch evaluation using Langfuse traces matched to Ground Truth entries.

#### Load Ground Truth Q & A

In [ ]:
import pandas as pd
import json
from datetime import datetime, timezone
from uuid import uuid4

from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall, AgentTrace, Message
from uaef.models.message import MessageRole

# Load ground truth data from Excel
excel_path = "data/ground-truth.xlsx"
df = pd.read_excel(excel_path)

# Convert Excel rows to JSON records for inspection
gt_json = df.to_dict(orient="records")

print(f"✓ Converted {len(gt_json)} rows to JSON")
print(f"✓ Loaded {len(df)} ground truth q/a from {excel_path}")
print(f"  Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()


#### Simulate Langfuse traces for each Ground Truth query

In production, you would fetch real traces from Langfuse matching each query. Here we simulate them.

In [ ]:
from uaef.adapters import LangfuseAdapter

adapter = LangfuseAdapter()
traces = []
ground_truths = []

for i, row in enumerate(gt_json):
    query, expected, context, expected_tools = parse_ground_truth_row(row)

    # Simulate a Langfuse trace for this query
    # In production: trace = langfuse_client.get_trace(trace_id)
    session_id = str(uuid4())
    langfuse_data = {
        "trace_id": str(uuid4()),
        "session_id": session_id,
        "name": f"batch-query-{i+1}",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "observations": [
            {
                "id": str(uuid4()),
                "type": "generation",
                "name": "llm-call",
                "input": {"role": "user", "content": query},
                "output": {"content": expected},
                "model": "anthropic.claude-3-sonnet",
                "usage": {"input": 30, "output": 50, "total": 80},
                "cost": 0.002,
                "start_time": datetime.now(timezone.utc).isoformat(),
                "end_time": datetime.now(timezone.utc).isoformat(),
            }
        ],
    }

    trace = adapter.transform_to_canonical(langfuse_data)
    traces.append(trace)
    ground_truths.append(GroundTruth(
        expected_output=expected,
        expected_tool_calls=expected_tools,
        context_documents=[context] if context else []
    ))

    label = f"'{query[:50]}...'" if len(query) > 50 else f"'{query}'"
    print(f"  [{i+1}/{len(gt_json)}] {label}")

print(f"\n✓ Prepared {len(traces)} Langfuse traces for evaluation")

#### Run evaluation on the agent traces

In [ ]:
# Batch evaluate all traces
batch_results = batch_evaluate(
    traces=traces,
    ground_truths=ground_truths,
    metrics=metrics,
    max_workers=4
)

print(f"{'='*50}")
print(f"BATCH RESULTS — LANGFUSE TRACES")
print(f"{'='*50}")
for i, r in enumerate(batch_results):
    status = "✓" if r.passed else "✗"
    print(f"  {status} Test {i+1}: {r.overall_score:.2f}")

avg = sum(r.overall_score for r in batch_results) / len(batch_results)
pr = sum(1 for r in batch_results if r.passed) / len(batch_results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")

#### Export batch evaluation results

In [ ]:
# Save evaluation results
from uaef.utils import save_metric_results

filepath = save_metric_results(batch_results, gt_json, prefix="langfuse_batch_results")
